# Generative Deep Learning

### Pick **one** of these topics.

The lecture notebooks should be used as starter code: a good first step is to remove all that is not necessary for your enquiry.

## 1. Text generation

The things that are important to understand for this topic are:
- the dataset processing and tokenization pipeline.
- the baseline computation (counting tokens -> turning that into probabilities -> finding the highest one).
- the model architecture.
- the sampling strategies.

The first off-the-shelf experiment (which is also fun) is to try building different language models. There are tens of thousands of texts in [Project Gutenberg](https://www.gutenberg.org). There is small but wide range of texts (and other data) [here](https://introcs.cs.princeton.edu/java/data/). Transformers are excellent models for any kind of sequence, so it can be interesting to see the results on e.g. chess moves, genome sequences, computer code, etc. The usual trade-off is: you can get nice but brittle results fast with less text, and any significant progress in generation will require scaling (the amount of text and the size of your model).

Another interesting experiment is to add special markers into your dataset, e.g. a special character, like `❡`, into the dataset before training, that allows you to control the behaviour of the trained net: if that character is used as an end point (like `<|endoftext|>`, that would be needed to be added to the tokenizer), splitting the dataset into parts, then in your generation loop that would allow you to stop generation, for instance, once that specific character has been produced. This is the logic behind chatbots (**supervised fine-tuning**): if you have markers for the end of a reply/question by a user or by a bot, and you have a dataset of that, then you can train your model to output the 'end of reply' token after it has answered your question, which produces the _effect_ of conversation.

Another approach is to think of what to do with a trained model: chapter 16 has a `generate` function that displays the text gradually. Can you implement a typewriter effect when generating, perhaps adding a bit of randomness, using `time.sleep()` in the loop, to make it a bit irregular?

You can find the pre-trained [Shakespeare model here](https://drive.google.com/file/d/1FYX2-j8TV1st6yUJ4HCSLAvCi1cOcA3i/view?usp=sharing), and the [Mini-C4 one here](https://drive.google.com/file/d/1y7HFXOF26_SywaRcCpe11FfFJ0Adf5Bq/view?usp=sharing).

#### Extra

Apart from `temperature` and `top_k` (sampling only from the most probable `k` logits), other common sampling techniques include `top_p`, nucleus sampling, which samples from the logits that in total amount to a certain probability mass `p` (e.g. `0.9`). The way it is done is: (arg)sorting the probabilities, then accumulate probabilities using `cumsum`, selecting the logits up to `p`, then applying the softmax on those only and sampling (see [here](https://huggingface.co/blog/how-to-generate) for a more complete discussion). It can be a nice challenge to implement this last one, but [Keras Hub also has various samplers](https://keras.io/keras_hub/api/samplers/).

## 2. VAEs

The things that are important to understand for this topic are:
- the model architecture (encoder/decoder, transposed convolutions)
- the sampling step (encoder predicting a mean and log variance, a sampler using that to sample a random point from a Gaussian)
- the concept of latent space and techniques to generate images.

There are two main alleyways when approaching this particular model: training, and sampling (given the pre-trained models available below, you can start in any order):

1. Training.

   - The key hyperparameter for the VAE is the dimensionality of the latent space. In Chapter 17 we pick 2, but it would be interesting to compare that with 1, 3, etc. (There's an example of a 5D VAE [here]((https://github.com/jchwenger/DLWP/blob/main/lectures/09.more/chapter17_image-generation.VAE-metrics.ipynb)), pre-trained model available below.
   - Instead of MNIST, it is possible to train on FashionMNIST, or the 200,000 celebrity portraits in the free [celebA dataset](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html) (also on [Kaggle](https://www.kaggle.com/jessicali9530/celeba-dataset)). Any image dataset will work (you may have to remove the labels), including dataset of synthetic images! It might even be interesting to try and combine two datasets (say, MNIST + FashionMNIST, since they are already in the same format), and see the results?
   - If you are interested in this topic for your CW, I recommend that you look at the notebook [`lectures/09.more/chapter17_image-generation.VAE-metrics.ipynb`](https://github.com/jchwenger/DLWP/blob/main/lectures/09.more/chapter17_image-generation.VAE-metrics.ipynb) (which has a discussion of baselines), and consider importing the KID metric from [`lectures/09.more/chapter17_image-generation.Diffusion-metrics.ipynb`](https://github.com/jchwenger/DLWP/blob/main/lectures/09.more/chapter17_image-generation.Diffusion-metrics.ipynb), since that metric works for any image generator.

2. Inference.

   - Another line of research is to explore and control sampling from the latent space. In the lecture code, we have a few examples already:
     - how to encode and decode data;
     - how to sample a random point in the latent space and generate a new image;
     - how to interpolate between two points using `lerp` and `slerp`;
     - how to create a grid of numbers linearly distributed between -1 and 1. (In the [metrics notebook](https://github.com/jchwenger/DLWP/blob/main/lectures/09.more/chapter17_image-generation.VAE-metrics.ipynb), there is also a modified grid function that creates a grid on chosen dimensions only. It is more difficult to explore higher dimensions! One could imagine also just modifying this plot function to do a much more refined interpolation (many more steps). And there might be smarter ways of going about exploring the 5D latent space.

You can find here [the 2D MNIST VAE (& Oxford Flowers diffusion model)](https://drive.google.com/file/d/1v-tA6NKSQD83hhWJd1t8_9Jth1zygkz7/view?usp=sharing) and here a [5D MNIST VAE](https://drive.google.com/file/d/1jGyqi2aIDMMkRkXYhgq672Spp3I9XPFL/view?usp=sharing).

#### Extra

It is also possible to isolate specific, meaningful vectors, and use vector maths to manipulate images. The method is the following:
  - gather a number of pictures with and without a feature (in the canonical example, with faces: with/without smile), the more the better;
  - encode all these into latent vectors using the Encoder;
  - take the mean of the latent vectors for each group;
  - subtract one by the other to obtain the vector representing the feature!
        
Finally, you can use that vector to modify an image, like so:
  - encode the image of choice using the Encoder;
  - subtract the feature vector from the image latent vector (literally `image_modified_z = image_z + feature_z`;
  - use the modified image latent vector and the Decoder to generate an image without the feature!

[This notebook](https://github.com/eduhrami/Hands-On-Image-Generation-with-TensorFlow-2.0/blob/master/Chapter02/ch2_vae_faces.ipynb) implements some of these (see also [this file](https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/03_vae/03_vae_faces/vae_utils.py)).

## 3. Diffusion

The things that are important to understand for this topic are:
- the model architecture (U-Net)
- the concept of noise schedule (noising/denoising)
- the overall training procedure: noise the image analytically to create a sample, train the model to predict the noise to recover the original image, compare our computed noise with the model's prediction
- the generation strategy: start from Gaussian noise, predict the noise to be removed, and use that to create the image at the next step (mixing the noise we start with and the prediction using the amounts prescribed by our schedule).

Just like the VAE, it is possible to experiment with training or with inference (you can use the pre-trained models below, or look at the text-to-image models section in the book).

1. Training
   
   - The main line of experiment here is to train a diffusion model on another dataset (MNIST, FashionMNIST, CelebA, generative images, your own data, etc.).

    - One nice thing to implement is to modify the generation callback save one image during training every time it generates (you can vary the frequency). If you create the random vector used as starting point **once** when initializing the callback, rather than when generation happens, then you will basically be able to see exactly how your model changes over time. When training is done, you can even combine those images together to make a video (LLMs today can easily help you implement that, and there's an [example using `imageio` here](https://github.com/jchwenger/DMLCP/blob/main/notebooks/05_dcgan_training.ipynb)). Remember that it is not necessary always to plot a grid of images: you could also simplify the code to plot only one image at a time (using `plt` or `PIL`), which would allow you to save single images more easily.

    - If you're interested in pursuing this for your CW, I recommend having a look at the baselines and the metric (Kernel Inception Distance, KID) in the notebook [`lectures/09.more/chapter17_image-generation.Diffusion-metrics.ipynb`](https://github.com/jchwenger/DLWP/blob/main/lectures/09.more/chapter17_image-generation.Diffusion-metrics.ipynb).

2. Inference

    - In the same way as during training (see above), it could be possible to hack the generation function save the picture being denoised at each step, to see the evolution. Then, the exact same code for turning a folder of pictures into a GIF/video could be used!
    - The [book](https://deeplearningwithpython.io/chapters/chapter17_image-generation/#text-to-image-models) also has a section that couldn't be covered in the lecture, using pre-trained models – much fun to be had with positive/negative prompts, and even an example of interpolation between two textual prompts (interpolation in the latent space of text embeddings works the same as within the latent space of the VAE).

You can find here [the pre-trained Oxford Flowers diffusion model (& 2D MNIST VAE)](https://drive.google.com/file/d/1v-tA6NKSQD83hhWJd1t8_9Jth1zygkz7/view?usp=sharing).

#### Extra

One thing that could be really interesting to learn to implement is a *class-conditional U-Net*, that also takes in the label associated with this samples, so that it is possible to guide the generation process that way! The key idea is to treat the label as you would with an embedding, so use an `Embedding` layer that has vectors of the same size as the images (`W x H`), so that you can embed your class label, resize it to `(W,H)`, and add it to the image *as a channel*. Then, the only thing you need is to change the number of channels your model expects, and it will train just the same way. Here's a [tutorial](https://huggingface.co/learn/diffusion-course/en/unit2/3) that focusses on this (in PyTorch).